In [1]:


#1) Imports and paths

import os
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import signal
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = "data"   
TRAIN_CSV = os.path.join(ROOT, "train.csv")
TEST_CSV  = os.path.join(ROOT, "test.csv")
RAW_DIR   = os.path.join(ROOT, "data")

FEATURE_TRAIN_OUT = os.path.join(ROOT, "train_features_v2.csv")
FEATURE_TEST_OUT  = os.path.join(ROOT, "test_features_v2.csv")
SUBMISSION_OUT    = os.path.join(ROOT, "submit_motion_model.csv")

d:\PY\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


#Metadata cleaning
def clean_metadata(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    out = df.copy()

    # gender "0" -> Unknown
    out["gender"] = out["gender"].replace({0: "Unknown", "0": "Unknown"}).astype(str)

    # age 0 is almost certainly missing, not true age
    out["age_missing"] = (out["age"] == 0).astype(int)
    out["age"] = out["age"].replace({0: np.nan}).astype(float)

    out["patient_off_on"] = out["patient_off_on"].astype(str)
    out["doctor_diagnosis_0_5"] = out["doctor_diagnosis_0_5"].astype(float)

    if is_train:
        out["folder_path"] = out["folder_path"].astype(str)

    return out


train = clean_metadata(pd.read_csv(TRAIN_CSV), is_train=True)
test  = clean_metadata(pd.read_csv(TEST_CSV), is_train=False)

print(train.shape, test.shape)
display(train.head())

(833, 7) (357, 6)


,data_file_name,folder_path,gender,age,patient_off_on,doctor_diagnosis_0_5,age_missing
0,raw_data_d786d645-db38-11ec-b494-e82aea2c97f4.csv,Kinetic tremor,Male,52.0,off,1.0,0
1,raw_data_bdcba44f-0d6a-11ed-8857-b6da2cf29e9d.csv,Postural tremor,Male,78.0,on,0.0,0
2,raw_data_750c0f09-b09a-11ec-9699-58a023d3f6d9.csv,Postural tremor,Male,71.0,on,0.0,0
3,raw_data_d90846c3-3969-11ed-a96d-b469216ca443.csv,Fist,Male,23.0,off,1.0,0
4,raw_data_c27fbeb3-1882-11ed-95c1-b469216ca443.csv,Kinetic tremor,Male,23.0,off,2.0,0


In [3]:
LANDMARKS = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_FINGER_MCP", "INDEX_FINGER_PIP", "INDEX_FINGER_DIP", "INDEX_FINGER_TIP",
    "MIDDLE_FINGER_MCP", "MIDDLE_FINGER_PIP", "MIDDLE_FINGER_DIP", "MIDDLE_FINGER_TIP",
    "RING_FINGER_MCP", "RING_FINGER_PIP", "RING_FINGER_DIP", "RING_FINGER_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]

TIPS = [
    "THUMB_TIP",
    "INDEX_FINGER_TIP",
    "MIDDLE_FINGER_TIP",
    "RING_FINGER_TIP",
    "PINKY_TIP",
]

ADJ_TIP_PAIRS = [
    ("THUMB_TIP", "INDEX_FINGER_TIP"),
    ("INDEX_FINGER_TIP", "MIDDLE_FINGER_TIP"),
    ("MIDDLE_FINGER_TIP", "RING_FINGER_TIP"),
    ("RING_FINGER_TIP", "PINKY_TIP"),
]

In [4]:


#Timestamp cleaning and segmentation
'''
preprocessing.
- detect resets (dt <= 0)
- detect giant gaps
- split the recording into continuous segments
- rebuild a local clean time inside each segment
- extract features segment-wise
- aggregate them back into one feature row per file
This avoids letting a bad 988-frame file with huge empty stretches dominate the features.
'''



def split_into_segments(
    df: pd.DataFrame,
    time_col: str = "TIME",
    min_frames: int = 20,
    gap_abs_seconds: float = 0.50,
    gap_mult: float = 8.0,
):
    """
    Split a raw file into continuous segments using:
      - timestamp reversal / reset (dt <= 0)
      - giant time jump (dt > max(gap_abs_seconds, gap_mult * median_positive_dt))

    Returns:
      kept_segments: list[pd.DataFrame]
      qc: dict of quality-control stats
    """
    out_segments = []

    if time_col not in df.columns:
        seg = df.copy()
        seg["_segment_id"] = 0
        seg["_segment_local_time"] = np.arange(len(seg), dtype=float)
        qc = {
            "raw_n_rows": len(df),
            "has_time": 0,
            "has_reset": 0,
            "has_large_gap": 0,
            "n_negative_dt": 0,
            "n_large_gaps": 0,
            "median_dt_raw": np.nan,
            "gap_threshold": np.nan,
            "n_segments_total": 1,
            "n_segments_kept": 1,
            "largest_segment_frac": 1.0,
        }
        return [seg], qc

    t = pd.to_numeric(df[time_col], errors="coerce").to_numpy(dtype=float)

    if len(t) <= 1:
        seg = df.copy()
        seg["_segment_id"] = 0
        seg["_segment_local_time"] = np.arange(len(seg), dtype=float)
        qc = {
            "raw_n_rows": len(df),
            "has_time": 1,
            "has_reset": 0,
            "has_large_gap": 0,
            "n_negative_dt": 0,
            "n_large_gaps": 0,
            "median_dt_raw": np.nan,
            "gap_threshold": np.nan,
            "n_segments_total": 1,
            "n_segments_kept": 1,
            "largest_segment_frac": 1.0,
        }
        return [seg], qc

    dt = np.diff(t)
    pos_dt = dt[np.isfinite(dt) & (dt > 0)]

    median_dt = np.median(pos_dt) if len(pos_dt) else (1.0 / 20.0)
    gap_threshold = max(gap_abs_seconds, gap_mult * median_dt)

    break_after = (~np.isfinite(dt)) | (dt <= 0) | (dt > gap_threshold)

    seg_id = np.zeros(len(df), dtype=int)
    seg_id[1:] = np.cumsum(break_after)

    all_lengths = []
    for sid in np.unique(seg_id):
        seg = df.loc[seg_id == sid].copy()
        all_lengths.append(len(seg))

        if len(seg) < min_frames:
            continue

        seg["_segment_id"] = sid
        seg["_segment_local_time"] = np.arange(len(seg), dtype=float) * median_dt
        out_segments.append(seg)

    largest_frac = (max(all_lengths) / len(df)) if len(all_lengths) else np.nan

    qc = {
        "raw_n_rows": len(df),
        "has_time": 1,
        "has_reset": int(np.any(dt <= 0)),
        "has_large_gap": int(np.any(dt > gap_threshold)),
        "n_negative_dt": int(np.sum(dt <= 0)),
        "n_large_gaps": int(np.sum(dt > gap_threshold)),
        "median_dt_raw": float(median_dt),
        "gap_threshold": float(gap_threshold),
        "n_segments_total": int(len(np.unique(seg_id))),
        "n_segments_kept": int(len(out_segments)),
        "largest_segment_frac": float(largest_frac),
        "raw_time_start": float(np.nanmin(t)) if np.isfinite(t).any() else np.nan,
        "raw_time_end": float(np.nanmax(t)) if np.isfinite(t).any() else np.nan,
        "raw_duration": float(np.nanmax(t) - np.nanmin(t)) if np.isfinite(t).any() else np.nan,
    }

    if len(out_segments) == 0:
        # fallback: keep everything in frame order with synthetic time
        seg = df.copy()
        seg["_segment_id"] = 0
        seg["_segment_local_time"] = np.arange(len(seg), dtype=float) * median_dt
        out_segments = [seg]
        qc["n_segments_kept"] = 1
        qc["largest_segment_frac"] = 1.0

    return out_segments, qc

In [5]:


#Geometry helpers
def get_xyz(df: pd.DataFrame, name: str) -> np.ndarray:
    cols = [f"{name}.x", f"{name}.y", f"{name}.z"]
    return df[cols].to_numpy(dtype=float)

def safe_norm(x: np.ndarray, axis: int = -1, eps: float = 1e-8) -> np.ndarray:
    return np.sqrt(np.sum(x * x, axis=axis) + eps)

def unit_vector(x: np.ndarray, axis: int = -1, eps: float = 1e-8) -> np.ndarray:
    return x / np.expand_dims(safe_norm(x, axis=axis, eps=eps), axis=axis)

def angle_between_vectors(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    a_u = unit_vector(a, eps=eps)
    b_u = unit_vector(b, eps=eps)
    cosang = np.sum(a_u * b_u, axis=1)
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.arccos(cosang)

def weighted_nanmean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights)
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])

In [6]:


# 1D signal summaries
'''
We use:
- basic descriptive stats
- movement roughness
- zero-crossing of centered signal
- dominant frequency
- spectral entropy
- bandpowers

'''
from scipy.integrate import trapezoid

def bandpower_from_psd(freqs, psd, lo, hi):
    mask = (freqs >= lo) & (freqs < hi)
    if mask.sum() == 0:
        return 0.0
    return float(trapezoid(psd[mask], freqs[mask]))

def summarize_1d_signal(x: np.ndarray, dt: float, prefix: str) -> dict:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    out = {}
    if len(x) == 0:
        keys = [
            "mean", "std", "min", "max", "median", "iqr", "rms", "range",
            "madiff", "zcr", "dom_freq", "spec_entropy",
            "bp_0_3", "bp_3_6", "bp_6_9", "bp_9_12"
        ]
        return {f"{prefix}__{k}": np.nan for k in keys}

    centered = x - np.median(x)

    out[f"{prefix}__mean"] = float(np.mean(x))
    out[f"{prefix}__std"] = float(np.std(x))
    out[f"{prefix}__min"] = float(np.min(x))
    out[f"{prefix}__max"] = float(np.max(x))
    out[f"{prefix}__median"] = float(np.median(x))
    out[f"{prefix}__iqr"] = float(np.percentile(x, 75) - np.percentile(x, 25))
    out[f"{prefix}__rms"] = float(np.sqrt(np.mean(x**2)))
    out[f"{prefix}__range"] = float(np.max(x) - np.min(x))
    out[f"{prefix}__madiff"] = float(np.mean(np.abs(np.diff(x)))) if len(x) > 1 else 0.0
    out[f"{prefix}__zcr"] = float(np.mean(centered[:-1] * centered[1:] < 0)) if len(x) > 1 else 0.0

    if dt is not None and np.isfinite(dt) and dt > 0 and len(x) >= 16:
        fs = 1.0 / dt
        freqs, psd = signal.welch(centered, fs=fs, nperseg=min(256, len(centered)))
        psd = np.asarray(psd, dtype=float)

        if len(freqs) > 1 and np.sum(psd[1:]) > 0:
            idx = int(np.argmax(psd[1:]) + 1)
            out[f"{prefix}__dom_freq"] = float(freqs[idx])

            p = psd[1:] / np.sum(psd[1:])
            out[f"{prefix}__spec_entropy"] = float(-(p * np.log(p + 1e-12)).sum())

            out[f"{prefix}__bp_0_3"] = bandpower_from_psd(freqs, psd, 0.0, 3.0)
            out[f"{prefix}__bp_3_6"] = bandpower_from_psd(freqs, psd, 3.0, 6.0)
            out[f"{prefix}__bp_6_9"] = bandpower_from_psd(freqs, psd, 6.0, 9.0)
            out[f"{prefix}__bp_9_12"] = bandpower_from_psd(freqs, psd, 9.0, 12.0)
        else:
            out[f"{prefix}__dom_freq"] = np.nan
            out[f"{prefix}__spec_entropy"] = np.nan
            out[f"{prefix}__bp_0_3"] = np.nan
            out[f"{prefix}__bp_3_6"] = np.nan
            out[f"{prefix}__bp_6_9"] = np.nan
            out[f"{prefix}__bp_9_12"] = np.nan
    else:
        out[f"{prefix}__dom_freq"] = np.nan
        out[f"{prefix}__spec_entropy"] = np.nan
        out[f"{prefix}__bp_0_3"] = np.nan
        out[f"{prefix}__bp_3_6"] = np.nan
        out[f"{prefix}__bp_6_9"] = np.nan
        out[f"{prefix}__bp_9_12"] = np.nan

    return out

In [7]:


# Cycle features
'''
This is borrowed in spirit from newer PD hand-movement work:
- amplitude
- cycle duration
- speed / rhythm
- interruption count
'''

def robust_smooth(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if len(x) < 9:
        return x.copy()
    win = min(len(x) if len(x) % 2 == 1 else len(x) - 1, 21)
    if win < 5:
        return x.copy()
    return signal.savgol_filter(x, window_length=win, polyorder=2)

def cycle_features(x: np.ndarray, dt: float, prefix: str) -> dict:
    x = np.asarray(x, dtype=float)
    out = {
        f"{prefix}__n_peaks": np.nan,
        f"{prefix}__cycle_dur_mean": np.nan,
        f"{prefix}__cycle_dur_std": np.nan,
        f"{prefix}__cycle_dur_slope": np.nan,
        f"{prefix}__cycle_amp_mean": np.nan,
        f"{prefix}__cycle_amp_std": np.nan,
        f"{prefix}__cycle_amp_slope": np.nan,
        f"{prefix}__interruptions": np.nan,
    }

    if len(x) < 20 or dt is None or not np.isfinite(dt) or dt <= 0:
        return out

    xs = robust_smooth(x)
    fs = 1.0 / dt

    min_peak_dist = max(3, int(0.15 * fs))
    prominence = max(1e-4, 0.20 * np.std(xs))

    peaks, _ = signal.find_peaks(xs, distance=min_peak_dist, prominence=prominence)
    troughs, _ = signal.find_peaks(-xs, distance=min_peak_dist, prominence=prominence)

    out[f"{prefix}__n_peaks"] = int(len(peaks))

    if len(peaks) >= 2:
        cycle_durs = np.diff(peaks) * dt
        out[f"{prefix}__cycle_dur_mean"] = float(np.mean(cycle_durs))
        out[f"{prefix}__cycle_dur_std"] = float(np.std(cycle_durs))
        out[f"{prefix}__interruptions"] = int(np.sum(cycle_durs > 2.0 * np.median(cycle_durs)))

        if len(cycle_durs) >= 2:
            out[f"{prefix}__cycle_dur_slope"] = float(
                np.polyfit(np.arange(len(cycle_durs)), cycle_durs, 1)[0]
            )

        amps = []
        for lp, rp in zip(peaks[:-1], peaks[1:]):
            mid_troughs = troughs[(troughs > lp) & (troughs < rp)]
            if len(mid_troughs) == 0:
                continue
            trough_val = float(np.min(xs[mid_troughs]))
            peak_val = float(max(xs[lp], xs[rp]))
            amps.append(peak_val - trough_val)

        if len(amps) > 0:
            amps = np.asarray(amps, dtype=float)
            out[f"{prefix}__cycle_amp_mean"] = float(np.mean(amps))
            out[f"{prefix}__cycle_amp_std"] = float(np.std(amps))
            if len(amps) >= 2:
                out[f"{prefix}__cycle_amp_slope"] = float(
                    np.polyfit(np.arange(len(amps)), amps, 1)[0]
                )

    return out

In [8]:


# Segment-level motion features
'''
Main feature engineering:
- wrist-centered, palm-scaled coordinates
- distance signals
- pose-change / motion-energy signals
- palm normal orientation and rotation speed
- cycle features and spectral features
'''

def extract_segment_features(seg: pd.DataFrame) -> dict:
    feats = {}

    t = seg["_segment_local_time"].to_numpy(dtype=float)
    dt = float(np.median(np.diff(t))) if len(t) > 1 else np.nan

    pts = {lm: get_xyz(seg, lm) for lm in LANDMARKS}

    wrist = pts["WRIST"]

    # translation invariance: wrist-centered
    rel = {lm: pts[lm] - wrist for lm in LANDMARKS}

    # scale invariance: palm size
    palm_scale = (
        safe_norm(rel["INDEX_FINGER_MCP"], axis=1) +
        safe_norm(rel["PINKY_MCP"], axis=1)
    ) / 2.0
    palm_scale = np.where(np.isfinite(palm_scale) & (palm_scale > 1e-6), palm_scale, np.nanmedian(palm_scale))
    palm_scale = np.where(np.isfinite(palm_scale) & (palm_scale > 1e-6), palm_scale, 1.0)

    rel_norm = {lm: rel[lm] / palm_scale[:, None] for lm in LANDMARKS}

    # 1) clinically intuitive distances
    thumb_index_dist = safe_norm(rel_norm["THUMB_TIP"] - rel_norm["INDEX_FINGER_TIP"], axis=1)
    fingertip_wrist_dists = np.column_stack([safe_norm(rel_norm[lm], axis=1) for lm in TIPS])
    mean_fingertip_wrist = np.mean(fingertip_wrist_dists, axis=1)

    fingertip_spread = np.mean(
        np.column_stack([
            safe_norm(rel_norm[a] - rel_norm[b], axis=1) for a, b in ADJ_TIP_PAIRS
        ]),
        axis=1,
    )

    index_tip_mcp_dist = safe_norm(rel_norm["INDEX_FINGER_TIP"] - rel_norm["INDEX_FINGER_MCP"], axis=1)
    middle_tip_mcp_dist = safe_norm(rel_norm["MIDDLE_FINGER_TIP"] - rel_norm["MIDDLE_FINGER_MCP"], axis=1)

    # 2) palm orientation for pronation/supination
    palm_v1 = rel_norm["INDEX_FINGER_MCP"]
    palm_v2 = rel_norm["PINKY_MCP"]
    palm_normal = np.cross(palm_v1, palm_v2)
    palm_normal = unit_vector(palm_normal)

    palm_nx = palm_normal[:, 0]
    palm_ny = palm_normal[:, 1]
    palm_nz = palm_normal[:, 2]

    palm_rot_speed = np.zeros(len(seg), dtype=float)
    if len(seg) > 1:
        palm_rot_speed[1:] = angle_between_vectors(palm_normal[:-1], palm_normal[1:]) / max(dt, 1e-8)

    # 3) pose-change / motion energy
    tip_stack = np.stack([rel_norm[lm] for lm in TIPS], axis=1)          # [T, 5, 3]
    all_stack = np.stack([rel_norm[lm] for lm in LANDMARKS], axis=1)     # [T, 21, 3]

    tip_motion_energy = np.zeros(len(seg), dtype=float)
    pose_motion_energy = np.zeros(len(seg), dtype=float)

    if len(seg) > 1:
        tip_motion_energy[1:] = np.mean(safe_norm(np.diff(tip_stack, axis=0), axis=2), axis=1) / max(dt, 1e-8)
        pose_motion_energy[1:] = np.mean(safe_norm(np.diff(all_stack, axis=0), axis=2), axis=1) / max(dt, 1e-8)

    # 4) angle signal
    thumb_vec = rel_norm["THUMB_TIP"]
    index_vec = rel_norm["INDEX_FINGER_TIP"]
    thumb_index_angle = angle_between_vectors(thumb_vec, index_vec)

    # summaries
    signals = {
        "thumb_index_dist": thumb_index_dist,
        "mean_fingertip_wrist": mean_fingertip_wrist,
        "fingertip_spread": fingertip_spread,
        "index_tip_mcp_dist": index_tip_mcp_dist,
        "middle_tip_mcp_dist": middle_tip_mcp_dist,
        "palm_nx": palm_nx,
        "palm_ny": palm_ny,
        "palm_nz": palm_nz,
        "palm_rot_speed": palm_rot_speed,
        "tip_motion_energy": tip_motion_energy,
        "pose_motion_energy": pose_motion_energy,
        "thumb_index_angle": thumb_index_angle,
    }

    for name, x in signals.items():
        feats.update(summarize_1d_signal(x, dt=dt, prefix=name))

    # cycle features on the most useful signals
    feats.update(cycle_features(thumb_index_dist, dt=dt, prefix="thumb_index_dist"))
    feats.update(cycle_features(fingertip_spread, dt=dt, prefix="fingertip_spread"))
    feats.update(cycle_features(palm_rot_speed, dt=dt, prefix="palm_rot_speed"))

    # segment QC
    feats["segment_n_rows"] = len(seg)
    feats["segment_duration"] = float(t[-1] - t[0]) if len(t) > 1 else 0.0
    feats["segment_dt"] = dt

    return feats

In [9]:



# Aggregate multiple segments back to one row
def aggregate_segment_features(segment_feature_dicts, segment_weights):
    keys = sorted(set().union(*[d.keys() for d in segment_feature_dicts]))
    out = {}

    for k in keys:
        vals = [d.get(k, np.nan) for d in segment_feature_dicts]
        out[k] = weighted_nanmean(vals, segment_weights)

    return out

In [10]:


# File-level featurization
def featurize_file(file_path: str) -> dict:
    df = pd.read_csv(file_path)

    segments, qc = split_into_segments(df)

    seg_feats = []
    seg_weights = []

    for seg in segments:
        try:
            f = extract_segment_features(seg)
            seg_feats.append(f)
            seg_weights.append(len(seg))
        except Exception as e:
            warnings.warn(
                f"Feature extraction failed for segment in {os.path.basename(file_path)}: {e}"
            )

    if len(seg_feats) == 0:
        base = {}
        base["feature_extraction_failed"] = 1
    else:
        base = aggregate_segment_features(seg_feats, seg_weights)
        base["feature_extraction_failed"] = 0

    base.update(qc)
    base["data_file_name"] = os.path.basename(file_path)
    return base

In [11]:


# Build train/test feature tables
def build_feature_table(meta_df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    rows = []

    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
        fname = row["data_file_name"]
        fpath = os.path.join(RAW_DIR, fname)

        feats = featurize_file(fpath)

        # append metadata
        feats["gender"] = row["gender"]
        feats["age"] = row["age"]
        feats["age_missing"] = row["age_missing"]
        feats["patient_off_on"] = row["patient_off_on"]
        feats["doctor_diagnosis_0_5"] = row["doctor_diagnosis_0_5"]

        if is_train:
            feats["folder_path"] = row["folder_path"]

        rows.append(feats)

    return pd.DataFrame(rows)


# Build once, then cache
train_feat = build_feature_table(train, is_train=True)
test_feat  = build_feature_table(test, is_train=False)

print(train_feat.shape, test_feat.shape)
print(train_feat["feature_extraction_failed"].value_counts(dropna=False))
display(train_feat.head())

100%|██████████| 357/357 [00:18<00:00, 18.82it/s]

(833, 241) (357, 240)
feature_extraction_failed
0    833
Name: count, dtype: int64


,fingertip_spread__bp_0_3,fingertip_spread__bp_3_6,fingertip_spread__bp_6_9,fingertip_spread__bp_9_12,fingertip_spread__cycle_amp_mean,fingertip_spread__cycle_amp_slope,fingertip_spread__cycle_amp_std,fingertip_spread__cycle_dur_mean,fingertip_spread__cycle_dur_slope,fingertip_spread__cycle_dur_std,...,raw_time_start,raw_time_end,raw_duration,data_file_name,gender,age,age_missing,patient_off_on,doctor_diagnosis_0_5,folder_path
0,0.015300,0.000814,0.000761,0.000620,0.271117,-0.049744,0.059399,1.226679,-0.141373,0.243936,...,0.155905,20.971091,20.815186,raw_data_d786d645-db38-11ec-b494-e82aea2c97f4.csv,Male,52.0,0,off,1.0,Kinetic tremor
1,0.000100,0.000013,0.000010,0.000008,0.017486,-0.000665,0.009891,1.064351,-0.023399,0.541282,...,0.048939,20.986482,20.937543,raw_data_bdcba44f-0d6a-11ed-8857-b6da2cf29e9d.csv,Male,78.0,0,on,0.0,Postural tremor
2,0.000116,0.000012,0.000007,0.000009,0.028388,0.002429,0.018507,2.454217,-0.308105,1.518682,...,0.051641,20.991918,20.940277,raw_data_750c0f09-b09a-11ec-9699-58a023d3f6d9.csv,Male,71.0,0,on,0.0,Postural tremor
3,0.010976,0.009689,0.002396,0.000078,0.096216,-0.002053,0.057787,0.503324,0.000334,0.276668,...,0.478883,19.231309,18.752425,raw_data_d90846c3-3969-11ed-a96d-b469216ca443.csv,Male,23.0,0,off,1.0,Fist
4,0.007368,0.000485,0.000279,0.000046,0.215608,0.007731,0.064551,1.594863,-0.141859,0.790287,...,0.217937,20.497118,20.279180,raw_data_c27fbeb3-1882-11ed-95c1-b469216ca443.csv,Male,23.0,0,off,2.0,Kinetic tremor


In [12]:


# Benchmark models
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

RANDOM_STATE = 42

# 1) Prepare X, y
target_col = "folder_path"
drop_cols = ["data_file_name", target_col]

X = train_feat.drop(columns=drop_cols, errors="ignore").copy()
y = train_feat[target_col].copy()

# Explicitly enforce metadata dtypes
for c in ["gender", "patient_off_on"]:
    if c in X.columns:
        X[c] = X[c].astype("string")

for c in ["age", "age_missing", "doctor_diagnosis_0_5"]:
    if c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

# Optional: if any feature columns accidentally became strings,
# try to coerce them back to numeric unless they are known categoricals
known_cat = {"gender", "patient_off_on"}
for c in X.columns:
    if c not in known_cat:
        if pd.api.types.is_object_dtype(X[c]) or pd.api.types.is_string_dtype(X[c]):
            X[c] = pd.to_numeric(X[c], errors="ignore")

# 2) Robust column selection
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical columns:", cat_cols)
print("Number of numeric columns:", len(num_cols))

# Sanity check: look for suspicious numeric columns
bad_num_cols = [c for c in num_cols if not pd.api.types.is_numeric_dtype(X[c])]
print("Non-numeric columns accidentally in num_cols:", bad_num_cols)

# 3) Preprocessing
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", ohe),
        ]), cat_cols),
    ],
    remainder="drop",
)

# 4) Models
logreg_pipe = Pipeline([
    ("prep", preprocess),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=4000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )),
])

rf_pipe = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=700,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
])

et_pipe = Pipeline([
    ("prep", preprocess),
    ("clf", ExtraTreesClassifier(
        n_estimators=900,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
])

models = {
    "LogReg": logreg_pipe,
    "RF": rf_pipe,
    "ExtraTrees": et_pipe,
}

try:
    from catboost import CatBoostClassifier

    cb_pipe = Pipeline([
        ("prep", preprocess),
        ("clf", CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric="TotalF1",
            depth=6,
            learning_rate=0.05,
            iterations=800,
            l2_leaf_reg=5,
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )),
    ])
    models["CatBoost"] = cb_pipe
except Exception as e:
    print("CatBoost not available, skipping it:", e)

# 5) CV setup
GROUP_COL = None
groups = train_feat[GROUP_COL] if (GROUP_COL is not None and GROUP_COL in train_feat.columns) else None

if groups is not None:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    fit_kwargs = {"groups": groups}
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    fit_kwargs = {}

scoring = {
    "accuracy": "accuracy",
    "balanced_acc": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
}

# 6) Benchmark
rows = []
for name, model in models.items():
    print(f"Running {name}...")
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score="raise",
        **fit_kwargs,
    )
    rows.append({
        "model": name,
        "acc_mean": scores["test_accuracy"].mean(),
        "acc_std": scores["test_accuracy"].std(),
        "bal_acc_mean": scores["test_balanced_acc"].mean(),
        "bal_acc_std": scores["test_balanced_acc"].std(),
        "macro_f1_mean": scores["test_macro_f1"].mean(),
        "macro_f1_std": scores["test_macro_f1"].std(),
        "weighted_f1_mean": scores["test_weighted_f1"].mean(),
        "weighted_f1_std": scores["test_weighted_f1"].std(),
    })

results_df = pd.DataFrame(rows).sort_values(
    ["macro_f1_mean", "bal_acc_mean", "acc_mean"],
    ascending=False
).reset_index(drop=True)

display(results_df)

Categorical columns: ['gender', 'patient_off_on']
Number of numeric columns: 237
Non-numeric columns accidentally in num_cols: []
CatBoost not available, skipping it: No module named 'catboost'
Running LogReg...
Running RF...
Running ExtraTrees...


,model,acc_mean,acc_std,bal_acc_mean,bal_acc_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,LogReg,0.966380,0.004835,0.962355,0.011935,0.960906,0.007828,0.966402,0.004790
1,ExtraTrees,0.961597,0.014515,0.943419,0.018706,0.949505,0.019134,0.961216,0.014555
2,RF,0.950776,0.021356,0.924828,0.027015,0.935150,0.027026,0.949915,0.021560


In [14]:


# Confusion matrix for the best model
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]

y_pred = cross_val_predict(
    best_model,
    X,
    y,
    cv=cv,
    n_jobs=1,
    **fit_kwargs,
)

print("Best model:", best_name)
print("Accuracy        :", accuracy_score(y, y_pred))
print("Balanced Acc    :", balanced_accuracy_score(y, y_pred))
print("Macro F1        :", f1_score(y, y_pred, average="macro"))
print("Weighted F1     :", f1_score(y, y_pred, average="weighted"))
print()
print(classification_report(y, y_pred, digits=4))

cm = confusion_matrix(y, y_pred, labels=sorted(y.unique()), normalize="true")
cm_df = pd.DataFrame(cm, index=sorted(y.unique()), columns=sorted(y.unique()))


Best model: LogReg
Accuracy        : 0.9663865546218487
Balanced Acc    : 0.962280763983179
Macro F1        : 0.9609107895349605
Weighted F1     : 0.9663647480302872

                                      precision    recall  f1-score   support

                      Finger tapping     0.9686    0.9439    0.9561       196
                                Fist     0.9700    0.9749    0.9724       199
                      Kinetic tremor     0.9636    0.9815    0.9725       162
                     Postural tremor     0.9767    0.9767    0.9767       215
Pronation and supination of the hand     0.9194    0.9344    0.9268        61

                            accuracy                         0.9664       833
                           macro avg     0.9597    0.9623    0.9609       833
                        weighted avg     0.9665    0.9664    0.9664       833



In [15]:
# ── Fig 1: Class distribution in training set ──────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

CLASS_COLORS = {
    'Finger tapping': '#4C72B0',
    'Fist': '#DD8452',
    'Kinetic tremor': '#55A868',
    'Postural tremor': '#C44E52',
    'Pronation and supination of the hand': '#8172B2',
}

class_counts = train_feat['folder_path'].value_counts()
labels = class_counts.index.tolist()
colors = [CLASS_COLORS.get(l, '#aaa') for l in labels]
short = ['Finger\ntapping', 'Fist', 'Kinetic\ntremor', 'Postural\ntremor', 'Pronation/\nsupination']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(short, class_counts.values, color=colors, edgecolor='white', linewidth=0.8, zorder=3)
ax.set_ylabel('Number of recordings', fontsize=11)
ax.set_title('Training Set Class Distribution', fontsize=13, fontweight='bold', pad=10)
ax.set_ylim(0, max(class_counts.values) * 1.18)
ax.yaxis.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax.set_axisbelow(True)
for bar, v in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(v),
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_class_dist.pdf', dpi=180, bbox_inches='tight')
plt.savefig('fig_class_dist.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig_class_dist.pdf / .png")


Saved fig_class_dist.pdf / .png


C:\Users\Admin\AppData\Local\Temp\ipykernel_3424\1196710965.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# ── Fig 2: Model comparison bar chart ──────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

metrics       = ['Accuracy', 'Balanced\nAccuracy', 'Macro F1', 'Weighted F1']
metric_keys   = [('acc_mean','acc_std'), ('bal_acc_mean','bal_acc_std'),
                 ('macro_f1_mean','macro_f1_std'), ('weighted_f1_mean','weighted_f1_std')]
model_colors  = {'CatBoost':'#2ecc71','LogReg':'#3498db','ExtraTrees':'#e67e22','RF':'#e74c3c'}

models_sorted = results_df['model'].tolist()
x = np.arange(len(metrics))
n = len(models_sorted)
bar_w = 0.18
fig, ax = plt.subplots(figsize=(9, 4.5))

for i, mname in enumerate(models_sorted):
    row = results_df[results_df['model'] == mname].iloc[0]
    means = [row[mk] for mk, _ in metric_keys]
    stds  = [row[sk] for _, sk in metric_keys]
    offset = (i - n/2 + 0.5) * bar_w
    bars = ax.bar(x + offset, means, bar_w, yerr=stds, capsize=3,
                  color=model_colors.get(mname, '#888'), label=mname,
                  edgecolor='white', linewidth=0.6, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0.88, 1.005)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.set_title('5-Fold CV Model Comparison (mean ± std)', fontsize=13, fontweight='bold', pad=10)
ax.legend(frameon=True, fontsize=9, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_model_comparison.pdf', dpi=180, bbox_inches='tight')
plt.savefig('fig_model_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig_model_comparison.pdf / .png")


Saved fig_model_comparison.pdf / .png


C:\Users\Admin\AppData\Local\Temp\ipykernel_3424\979989391.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# ── Fig 3: Confusion matrix heatmap ────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, y_pred, labels=sorted(y.unique()), normalize='true')
class_labels_short = ['Finger\nTapping', 'Fist', 'Kinetic\nTremor', 'Postural\nTremor', 'Pronation/\nSupination']

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Proportion', fontsize=10)

n_classes = len(class_labels_short)
ax.set_xticks(np.arange(n_classes))
ax.set_yticks(np.arange(n_classes))
ax.set_xticklabels(class_labels_short, fontsize=9)
ax.set_yticklabels(class_labels_short, fontsize=9)
ax.set_xlabel('Predicted label', fontsize=11, labelpad=8)
ax.set_ylabel('True label', fontsize=11, labelpad=8)
ax.set_title('CatBoost Normalised Confusion Matrix\n(OOF predictions, n=833)', fontsize=12, fontweight='bold', pad=10)

thresh = cm.max() / 2.0
for i in range(n_classes):
    for j in range(n_classes):
        color = 'white' if cm[i, j] > thresh else 'black'
        ax.text(j, i, f'{cm[i,j]:.3f}', ha='center', va='center',
                fontsize=9, color=color, fontweight='bold' if i == j else 'normal')

plt.tight_layout()
plt.savefig('fig_confusion_matrix.pdf', dpi=180, bbox_inches='tight')
plt.savefig('fig_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig_confusion_matrix.pdf / .png")


Saved fig_confusion_matrix.pdf / .png


C:\Users\Admin\AppData\Local\Temp\ipykernel_3424\294407670.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# ── Fig 4: Per-class precision / recall / F1 ───────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

labels_sorted = sorted(y.unique())
prec, rec, f1, supp = precision_recall_fscore_support(y, y_pred, labels=labels_sorted)

short_labels = ['Finger\nTapping', 'Fist', 'Kinetic\nTremor', 'Postural\nTremor', 'Pronation/\nSupination']
x = np.arange(len(labels_sorted))
bar_w = 0.25

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(x + bar_w, prec, bar_w, label='Precision', color='#3498db', edgecolor='white')
ax.barh(x,         rec,  bar_w, label='Recall',    color='#2ecc71', edgecolor='white')
ax.barh(x - bar_w, f1,   bar_w, label='F1-score',  color='#e74c3c', edgecolor='white')

ax.set_yticks(x)
ax.set_yticklabels(short_labels, fontsize=10)
ax.set_xlabel('Score', fontsize=11)
ax.set_xlim(0.85, 1.02)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.set_title('Per-Class Metrics — CatBoost (OOF)', fontsize=12, fontweight='bold', pad=10)
ax.legend(fontsize=9, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for i, (p, r, f) in enumerate(zip(prec, rec, f1)):
    ax.text(p + 0.001, i + bar_w, f'{p:.3f}', va='center', fontsize=7.5, color='#2c3e50')
    ax.text(r + 0.001, i,         f'{r:.3f}', va='center', fontsize=7.5, color='#2c3e50')
    ax.text(f + 0.001, i - bar_w, f'{f:.3f}', va='center', fontsize=7.5, color='#2c3e50')

plt.tight_layout()
plt.savefig('fig_per_class_metrics.pdf', dpi=180, bbox_inches='tight')
plt.savefig('fig_per_class_metrics.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig_per_class_metrics.pdf / .png")


Saved fig_per_class_metrics.pdf / .png


C:\Users\Admin\AppData\Local\Temp\ipykernel_3424\1258985619.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# ── Fig 5: Top-20 feature importances from best tree model ─────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Re-fit best model on full training data to extract importances
best_model.fit(X, y)

clf_step = best_model.named_steps['clf']
prep_step = best_model.named_steps['prep']

# Get feature names out of the preprocessor
try:
    feat_names = prep_step.get_feature_names_out()
except Exception:
    feat_names = None

# Extract feature importances, handling different model types
try:
    importances = clf_step.feature_importances_
except AttributeError:
    try:
        importances = clf_step.get_feature_importance()
    except AttributeError:
        if hasattr(clf_step, 'coef_'):
            # For linear models like LogisticRegression
            importances = np.abs(clf_step.coef_).mean(axis=0)
        else:
            raise ValueError("Cannot extract feature importances for this model")

if feat_names is not None and len(feat_names) == len(importances):
    # Strip sklearn prefixes like "num__" / "cat__"
    clean_names = [n.split('__', 1)[-1] if '__' in n else n for n in feat_names]
else:
    clean_names = [f'f{i}' for i in range(len(importances))]

idx = np.argsort(importances)[::-1][:20]
top_names = [clean_names[i] for i in idx]
top_vals  = importances[idx]
top_vals_norm = top_vals / top_vals.sum()

# Colour by feature group
def feat_color(name):
    if 'dom_freq' in name or 'spec_entropy' in name or 'bp_' in name:
        return '#8172B2'   # spectral
    if 'cycle' in name:
        return '#55A868'   # cycle
    if 'palm' in name or 'rot' in name:
        return '#C44E52'   # orientation
    if 'tip' in name or 'fingertip' in name or 'thumb' in name:
        return '#DD8452'   # fingertip
    return '#4C72B0'

colors = [feat_color(n) for n in top_names]

legend_patches = [
    plt.Rectangle((0,0),1,1, color='#8172B2', label='Spectral'),
    plt.Rectangle((0,0),1,1, color='#55A868', label='Cycle'),
    plt.Rectangle((0,0),1,1, color='#C44E52', label='Palm orientation'),
    plt.Rectangle((0,0),1,1, color='#DD8452', label='Fingertip/thumb'),
    plt.Rectangle((0,0),1,1, color='#4C72B0', label='Other'),
]

fig, ax = plt.subplots(figsize=(8, 6))
y_pos = np.arange(len(top_names))
ax.barh(y_pos, top_vals_norm, color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_names, fontsize=8.5)
ax.invert_yaxis()
ax.set_xlabel('Normalised importance', fontsize=11)
ax.set_title(f'Top-20 Feature Importances — {best_name}', fontsize=12, fontweight='bold', pad=10)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(handles=legend_patches, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('fig_feature_importance.pdf', dpi=180, bbox_inches='tight')
plt.savefig('fig_feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig_feature_importance.pdf / .png")

Saved fig_feature_importance.pdf / .png


C:\Users\Admin\AppData\Local\Temp\ipykernel_3424\462344884.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# ── Fig 6: Test-set prediction distribution ────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

CLASS_COLORS = {
    'Finger tapping': '#4C72B0',
    'Fist': '#DD8452',
    'Kinetic tremor': '#55A868',
    'Postural tremor': '#C44E52',
    'Pronation and supination of the hand': '#8172B2',
}

pred_counts = submission['pred'].value_counts()
short_test  = ['Finger\nTapping', 'Fist', 'Kinetic\nTremor', 'Postural\nTremor', 'Pronation/\nSupination']
full_labels = pred_counts.index.tolist()
colors = [CLASS_COLORS.get(l, '#aaa') for l in full_labels]

# also plot train distribution side by side
train_counts = train_feat['folder_path'].value_counts()
# align to same order as pred_counts
train_vals = [train_counts.get(l, 0) / train_counts.sum() for l in full_labels]
test_vals  = pred_counts.values / pred_counts.values.sum()

x = np.arange(len(full_labels))
bar_w = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - bar_w/2, train_vals, bar_w, label='Train (true)', color=colors, alpha=0.6, edgecolor='white')
ax.bar(x + bar_w/2, test_vals,  bar_w, label='Test (predicted)', color=colors, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(short_test, fontsize=10)
ax.set_ylabel('Proportion', fontsize=11)
ax.set_title('Class Distribution: Train (true) vs Test (predicted)', fontsize=12, fontweight='bold', pad=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig_test_distribution.pdf', dpi=180, bbox_inches='tight')
plt.savefig('fig_test_distribution.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig_test_distribution.pdf / .png")


Saved fig_test_distribution.pdf / .png


C:\Users\Admin\AppData\Local\Temp\ipykernel_3424\1835599930.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:

# safest: force test features to have the same columns and order as training X
X_test = test_feat[X.columns].copy()

test_pred = best_model.predict(X_test)

# flatten prediction to 1D
test_pred = np.asarray(test_pred).reshape(-1)

submission = pd.DataFrame({
    "path": test_feat["data_file_name"].to_numpy(),
    "pred": test_pred,
})

submission.to_csv(SUBMISSION_OUT, index=False)
print("Saved:", SUBMISSION_OUT)
display(submission.head())
print(submission.shape)

Saved: data\submit_motion_model.csv


,path,pred
0,raw_data_61fbeecd-8ef7-11ec-a78a-58a023d3f6d9.csv,Finger tapping
1,raw_data_41f438e7-8041-11ed-aa7b-e82aea2c97f4.csv,Postural tremor
2,raw_data_43b5ffcb-2dcc-11ed-a854-e82aea2c97f4.csv,Postural tremor
3,raw_data_28cfe921-2452-11ed-934f-e82aea2c97f4.csv,Finger tapping
4,raw_data_d6107ceb-2f57-11ed-8d23-c86000e163af.csv,Kinetic tremor


(357, 2)


In [22]:
print(type(test_pred))
print(np.asarray(test_pred).shape)
print(test_pred[:5])

<class 'numpy.ndarray'>
(357,)
['Finger tapping' 'Postural tremor' 'Postural tremor' 'Finger tapping'
 'Kinetic tremor']


In [23]:
print(submission.columns.tolist())
print(submission.shape)
print(submission["path"].nunique(), len(submission))
print(submission["pred"].value_counts())

['path', 'pred']
(357, 2)
357 357
pred
Finger tapping                          97
Postural tremor                         88
Fist                                    75
Kinetic tremor                          69
Pronation and supination of the hand    28
Name: count, dtype: int64
